# Module 6 - Alzheimer's: Adverse Reaction Profiles

Top reactions across the full Alzheimer's cohort, then broken down by drug class. Assumes `alzheimers_analysis` has been created by `01_alz_explore.ipynb`.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

db_path = r"C:\Users\palla\OneDrive\Documents\Coding Projects\FDA_FAERS\database\faers.db"
conn = sqlite3.connect(db_path)

## Step 1 - Top 20 reactions across the whole cohort

In [ ]:
top_reactions = pd.read_sql_query("""
    SELECT r.pt AS reaction,
           COUNT(DISTINCT r.primaryid) AS reports
    FROM reac r
    JOIN (SELECT DISTINCT primaryid FROM alzheimers_analysis) a
        ON r.primaryid = a.primaryid
    GROUP BY r.pt
    ORDER BY reports DESC
    LIMIT 20
""", conn)

plt.figure(figsize=(10, 7))
plt.barh(top_reactions['reaction'], top_reactions['reports'], color='steelblue')
plt.gca().invert_yaxis()
plt.xlabel('Distinct reports')
plt.title('Top 20 reactions - Alzheimer\'s cohort')
plt.tight_layout()
plt.show()

top_reactions

## Step 2 - Reaction x Class heatmap (%-normalized within class)

In [ ]:
# For every top-30 reaction, what % of each class's reports mentioned it?
# Column-normalization controls for the very different class cohort sizes.

reactions_x_class = pd.read_sql_query("""
    SELECT r.pt AS reaction,
           a.drug_class,
           COUNT(DISTINCT r.primaryid) AS reports
    FROM reac r
    JOIN alzheimers_analysis a ON a.primaryid = r.primaryid
    GROUP BY r.pt, a.drug_class
""", conn)

class_totals = pd.read_sql_query("""
    SELECT drug_class,
           COUNT(DISTINCT primaryid) AS total
    FROM alzheimers_analysis
    GROUP BY drug_class
""", conn).set_index('drug_class')['total']

wide = (reactions_x_class
        .pivot(index='reaction', columns='drug_class', values='reports')
        .fillna(0))

# Keep the top-30 reactions by row sum
wide['total'] = wide.sum(axis=1)
wide = wide.sort_values('total', ascending=False).head(30).drop(columns='total')

pct = wide.div(class_totals, axis=1) * 100

plt.figure(figsize=(9, 10))
plt.imshow(pct.values, aspect='auto', cmap='viridis')
plt.colorbar(label='% of class reports')
plt.xticks(range(len(pct.columns)), pct.columns, rotation=30, ha='right')
plt.yticks(range(len(pct.index)),   pct.index)
plt.title('Top 30 reactions x drug class (% of class reports)')
plt.tight_layout()
plt.show()

## Step 3 - Reactions unique or dominant to each class

In [ ]:
# For each class, rank reactions by 'class share' (this class's rate / average across other classes).
# Interpretation: values >> 1 mean the reaction is characteristic of this class.

other_rates = pct.copy()
for col in pct.columns:
    other_cols = [c for c in pct.columns if c != col]
    other_rates[col] = pct[other_cols].mean(axis=1)

dominance = (pct / other_rates.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

for cls in pct.columns:
    top = dominance[cls].dropna().sort_values(ascending=False).head(8)
    print(f'\n=== Top-dominant reactions for {cls} ===')
    print(top.round(2).to_string())